# Notebook 7 — Decision-Curve Analysis for Clinical AI

Companion to **Chapters 8 & 11**. Discrimination metrics (AUC) and calibration are not enough to tell whether a model is *clinically useful*. **Decision-curve analysis** (Vickers & Elkin, 2006) plots *net benefit* across thresholds and compares to the trivial "treat all" and "treat none" strategies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from code.clinical.risk_prediction import generate_synthetic_ehr, FEATURE_COLS
from code.clinical.decision_curve import decision_curve, informative_range
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
df = generate_synthetic_ehr(n_samples=5000, seed=0)
X_train, X_test, y_train, y_test = train_test_split(df[FEATURE_COLS], df['attempt'], test_size=0.4, random_state=0, stratify=df['attempt'])
model = CalibratedClassifierCV(LogisticRegression(max_iter=2000), cv=5, method='isotonic').fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
print('base rate:', round(float(y_test.mean()), 3))

In [ ]:
curve = decision_curve(y_test.values, proba, thresholds=np.linspace(0.005, 0.5, 100))
df_dca = pd.DataFrame([pt.__dict__ for pt in curve])
df_dca[['threshold', 'net_benefit_model', 'net_benefit_treat_all']].set_index('threshold').plot(figsize=(7, 4))
plt.axhline(0, color='k', lw=0.5)
plt.title('Decision Curve Analysis')
plt.ylabel('Net Benefit')
plt.xlabel('Threshold probability')
plt.show()
print('Informative threshold range:', informative_range(curve))

## Reflection

Within the informative threshold range the model is more useful than either baseline; outside it, deployment is questionable.

## Exercises

1. Halve the base rate and re-run. Does the informative range move?
2. Compare a calibrated and an uncalibrated model. Calibration matters for DCA — show this.
3. Pick a threshold a clinician would actually use. Defend it in terms of false-positive cost.